1. Load Model and Dependencies

In [ ]:
%pip install -q transformers accelerate bitsandbytes datasets pandas tqdm sae-lens huggingface_hub numpy

In [ ]:
import os
import io
import random
from tqdm.auto import tqdm
#---TORCH---
import torch
import torch.nn as nn
import torch.nn.functional as F
#---ANALYZE---
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
#---MODEL---
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
from sae_lens import SAE
from huggingface_hub import login
from huggingface_hub import snapshot_download
#---MISC---
from collections import Counter
from google.colab import drive
from google.colab import userdata


Set our random seed for reproducability.

In [ ]:
random.seed(2006)

Google Drive

We will be using Google Drive to store our models states. Training a CRL learner can take a lot of time, so it can be helpful to have checkpointing to save our progress and train in chunks.



In [ ]:
drive.mount('/content/drive')

# define and create our save directory
CHECKPOINT_DIR = '/content/drive/MyDrive/CRL_Learner/issue_spotting_prompt_experiment_newtrain_lr5e-6_entropyloss/'
if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2. Configure the model, SAE, and tasks benchmark for evaluation

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

model_id = "google/gemma-2-2b"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    token=hf_token
)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Below we load our SAE dictonary, which is needed to convert the residual stream of the trasnformer layer to the interpretable features.

In [ ]:
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gemma-scope-2b-pt-res-canonical",
    sae_id="layer_20/width_16k/canonical",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

/tmp/ipykernel_17129/2126259194.py:1: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, sparsity = SAE.from_pretrained(


We use constrained decoding to restrict the model to the only correct answers. This works on multiple choice tasks, though my not be needed for multi-token task.

In [ ]:
valid_tokens = [i[0] for i in tokenizer(["Yes", "No"], add_special_tokens=False).input_ids]

class AllowedTokensProcessor(LogitsProcessor):
    def __init__(self, allowed_tokens):
        self.allowed_tokens = torch.tensor(allowed_tokens)

    def __call__(self, input_ids, scores):
        # scores: (batch_size, vocab_size)
        mask = torch.full_like(scores, float("-inf"))
        mask[:, self.allowed_tokens.to(scores.device)] = 0
        return scores + mask

processors = LogitsProcessorList([
    AllowedTokensProcessor(valid_tokens)
])



The hook is our injection into the models thought process. For our purposes, this is where our actor is going to need to make choices about activated features. We only want to steer our features on the generated outputs which is the last token in our case.

We need to store the size of the residual stream and feature selection from our SAE for our models architecture.

In [ ]:
total_features = cfg_dict['d_sae']
model_dimensions = cfg_dict['d_in']

print(f"Total features available: {total_features}")
print(f"Model dimensions: {model_dimensions}")

Total features available: 16384
Model dimensions: 2304


LegalBench is consisted of 162 tasks, spreading across 6 categories. Issue spotting is one of these categories, and consists entirley of binary yes / no question to test a agents ability on identifying legal issues.

Each one of these tasks is comprised of samples, and we will need to download these samples to use a training and testing data for our agent. Issue spotting has 17 different categories, most derived from the learned hands dataset. All together they have 11,512 test samples to pull from.

In [ ]:
ISSUE_SPOTTING_TASK = [ "learned_hands_benefits", "learned_hands_business", "learned_hands_consumer", "learned_hands_courts",  "learned_hands_crime", "learned_hands_divorce", "learned_hands_domestic_violence", "learned_hands_employment", "learned_hands_education", "learned_hands_estates", "learned_hands_family", "learned_hands_health", "learned_hands_housing", "learned_hands_immigration", "learned_hands_torts", "learned_hands_traffic" ]

Here we load our benchmark dataset, LegalBench. LegalBench train splits only have 6 tasks in each, so instead of using that as our training data we will create our own training, validation, and test splits from the much larger test datasets from LegalBench.

In [ ]:
datasets = {}

for task in ISSUE_SPOTTING_TASK :
    # load_dataset is the datasets library we imported at the beginning
    new_dataset = load_dataset("nguha/legalbench", task, split="test")
    datasets[task] = new_dataset



Next, we need to "shuffle" our dataset so we can sample the train test and validation segments later. It would be inefficent to load the dataset twice and there are benefits in evaluation for using a map, so we can create indices to relate to tasks and store them in a list instead. The entries of the list are in form (task name, index in task dataset).

In [ ]:
global_indices = []
for task_name, dataset in datasets.items():
    for idx in range(len(dataset)):
        global_indices.append((task_name, idx))

# Shuffle the indices
random.shuffle(global_indices)


total_samples = len(global_indices)
train_end = int(total_samples * 0.6)
val_end = train_end + int(total_samples * 0.2)

3. The actor agent

Our actor is a part of the PPO implementation. We just use a 2 layer MLP with tanh activation. The actors job is to map the residual stream ( our state ) to SAE features ( our actions ). We train the network parameters in a wrapper defined later in the notebook. The only things that we need to pass throuhg to our actor is the dimenision of the residual stream ( in our case above thats 2304 ) and dimensions of our dictonary ( 16384 features )

In [ ]:
class Actor(nn.Module) :
    def __init__(self, d_res, d_sae) :
        super().__init__()

        # Architecture of the actor network
        self.net = nn.Sequential(
            nn.Linear(d_res, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, d_sae)
        )

        self.num_features = d_sae

    def forward(self, x):
        return self.net(x)

4. The critic agent

Our critic is similar to our actor, except the output dimensions is just 1 which represents the value of a state V(S).

In [ ]:
class Critic(nn.Module) :
    def __init__(self, d_res) :
        super().__init__()

        # Architecture of the actor network
        self.net = nn.Sequential(
            nn.Linear(d_res, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1) # one output for the value function
        )

    def forward(self, x):
        return self.net(x)

5. Adaptive Feature Masking

I will just quickly mention adaptive feature masking (AFM). This is a technique used to help the agent explore the options it is given. There are roughly 16k features for the agent to choose from as a action, so AFM retricts this action space and slowly expands it to relevant features. Our mask is initialized to all zeros, and when a feature is activated through a forward pass it get sets to 1. Then we apply this mask against our logits before softmax so we can filter out the irrelevant features and our agent has a more managable action space. As the agent encounters new prompts in training, new features will activate thus expanding this action space to only the relevant actions it can take. In this experiment I take a slightly different approach, only setting the mask to the currently activated features of the pass.

The function below takes in a list that represents sae feature activations and returns a list of tuples where the tuples are (activation_index, activation_value)

In [ ]:
def get_labeled_activations (activations_tensor) :
  activated_features = []
  # activations_tensor is expected to be [batch_size, total_features]
  # We process the feature vector for the first (and likely only) item in the batch.
  feature_vector = activations_tensor.squeeze(0) # Converts [1, 16384] to [16384]

  for i in range(feature_vector.shape[0]) : # Iterate through the elements of the 1D feature_vector
    if feature_vector[i].item() > 0 : # Use .item() to extract the scalar value for comparison
      activated_features.append((i, feature_vector[i].item())) # Use .item() for list storage

  return activated_features

Below I employ the use of two wrapper agents to ease the implementation of the neural networks and the residual stream intervention that will be applied. This also has a benefit of keeping the training loop simple, as all of our updates and training logic will be executed by our wrapper agents during the models forward pass.

6. CRL Wrapper Agent

Our CRLController is our main wrapper, and it contains both policy networks and our SAE dictonary. It also keeps our learned c co-efficent and expanded mask. This means that a CRLController will be layer specific, but modular when it comes to tasks. By being able to swap out our critic and actor networks we are able to train for different tasks without needing to write large updates to our training and steering large.

In [ ]:
class CRLController :
  def __init__(self, sae, actor, critic) :
    #---INTERNAL AGENTS---
    # refrences to agents and SAE dict
    self.sae = sae
    self.actor = actor
    self.critic = critic

    # Optimizers for our actor and critic
    self.actor_optimizer = torch.optim.Adam(actor.parameters(), lr=5e-6)
    self.critic_optimizer = torch.optim.Adam(critic.parameters(), lr=1e-4)

    # ---HYPERPARAMETERS---
    # clip_e defines how big our trust region is for PPO updates
    self.clip_e = 0.2
    self.ent_coef = 0.01

    # ---BEHAVIOUR FLAGS---
    self.steering_active = True
    self.training_active = True

    # ---CRL LEARNED VARIABLES---
    self.c = 50
    self.mask = torch.zeros(actor.num_features)

    # ---MEMORY BUFFER---
    self.memory_buffer = []

    # ---SELF TRACKING---
    self.critic_loss_history = []
    self.actor_loss_history = []

    # Used to view model decision making process for evaluation
    self.last_feature_selection = 0
    self.last_critic_estimation = 0

  # Wrapper functions to toggle off and on behaviour of CRLController
  def is_steering(self, active) :
    self.steering_active = active

  def is_training(self, active) :
    self.training_active = active

  # Return how many active features are in the mask
  def get_mask_size(self) :
    return self.mask.sum().item()

  # Use matplotlib to generate two graphs of actor and critic loss
  def generate_graphs(self) :
    # Set a clean style
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # 1. Critic Loss Plot
    ax1.plot(self.critic_loss_history, color='#2c3e50', linewidth=1.5)
    ax1.set_title('Critic Loss History', fontsize=12, fontweight='bold', pad=10)
    ax1.set_xlabel('Epoch / Iteration', fontsize=10)
    ax1.set_ylabel('Loss', fontsize=10)
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    # 2. Actor Loss Plot
    ax2.plot(self.actor_loss_history, color='#e74c3c', linewidth=1.5)
    ax2.set_title('Actor Loss History', fontsize=12, fontweight='bold', pad=10)
    ax2.set_xlabel('Epoch / Iteration', fontsize=10)
    ax2.set_ylabel('Loss', fontsize=10)
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    plt.tight_layout()

    # Save the plot to an in-memory byte buffer
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    plt.close(fig)  # Clear the figure memory immediately

    buf.seek(0)
    image_bytes = buf.getvalue()
    buf.close()

    return image_bytes

  # Train episode executes a policy update for both our actor and critic agents based on a given episode
  # It is expected that episode is extracted from our memory buffer, otherwise you will get key access errors
  def train (self) :

    # Convert scalar inputs to tensors and ensure state requires grad
    # `state` was saved from `x_t_squeezed.clone().detach()`. We need it to require grad for the current training networks.
    states = torch.stack([sample["state"] for sample in self.memory_buffer]).to(device=device) # Shape: [Batch, Features]
    states = states.view(len(self.memory_buffer), -1)

    chosen_features = torch.tensor([sample["chosen_feature"] for sample in self.memory_buffer], dtype=torch.long, device=device).view(-1)

    # We wrap our rewards and probs in a tensor so they can be passed to our loss functions
    old_log_probs = torch.tensor([sample["chosen_feature_log_prob"] for sample in self.memory_buffer], dtype=torch.float32, device=device).view(-1)
    rewards = torch.tensor([sample["reward"] for sample in self.memory_buffer], dtype=torch.float32, device=device).unsqueeze(-1) # Shape: [Batch, 1]
    advantages = torch.tensor([sample["advantage"] for sample in self.memory_buffer], dtype=torch.float32, device=device).view(-1)
    masks = torch.stack([torch.tensor(sample["mask"], dtype=torch.bool, device=device) for sample in self.memory_buffer])

    # --- CRITIC UPDATE ---
    # The critic should predict the target `reward`. Its input `state` must have requires_grad=True.
    current_critic_estimations = self.critic(states) # This will have shape [1, 1]
    critic_loss = F.mse_loss(current_critic_estimations, rewards) # reward now has shape [1,1] matching current critic estimation

    self.critic_loss_history.append(critic_loss.item())

    self.critic_optimizer.zero_grad()
    critic_loss.backward()
    torch.nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=0.5)
    self.critic_optimizer.step()

    # --- ACTOR UPDATE ---
    # We pass through our current actor network to use in our ratio
    actor_logits = self.actor(states) # state has requires_grad=True, so actor_logits will too

    # Clone the actor logits to alter them and compute log probs
    masked_logits = actor_logits.clone()
    masked_logits = torch.where(masks, masked_logits, torch.tensor(-1e9, device=device))
    dist = torch.distributions.Categorical(logits=masked_logits)
    entropy = dist.entropy().mean()

    # Compute our ratio r(t) based on new and old policy action selection
    # Chosen_feature is an int, it's fine for direct indexing here
    current_log_probs = dist.log_prob(chosen_features)
    ratios = torch.exp(current_log_probs - old_log_probs)
    surr1 = ratios * advantages
    surr2 = torch.clamp(ratios, 1.0 - self.clip_e, 1.0 + self.clip_e) * advantages

    actor_loss = -torch.min(surr1, surr2).mean() - (self.ent_coef * entropy)

    self.actor_loss_history.append(actor_loss.item())

    self.actor_optimizer.zero_grad()
    actor_loss.backward()
    torch.nn.utils.clip_grad_norm_(self.actor.parameters(), max_norm=0.5)
    self.actor_optimizer.step()


  # Apply steering intervention given flags and a residual stream
  # The function is called within the hook intervention of our forward pass
  # x_t represents the current residual stream
  def steer(self, x_t) :
    # Do nothing if turned off
    if not self.steering_active :
      return x_t

    # x_t comes in as shape [1,1,2034], squeeze to get residual stream alone
    # We store the original shape to unsqueeze before returning
    original_shape_x_t = x_t.shape
    x_t_squeezed = x_t.squeeze(1) # Squeeze the sequence length dimension

    # If we aren't training, we don't want to keep track of our grad graphs
    if not self.training_active :
      with torch.no_grad() :
        actor_logits = self.actor(x_t_squeezed)
        critic_logits = self.critic(x_t_squeezed)
    else :
      actor_logits = self.actor(x_t_squeezed)
      critic_logits = self.critic(x_t_squeezed)

    # Get the critic estimation for our state value
    v_s = critic_logits.detach().item()
    self.last_critic_estimation = v_s

    # Pass our activations through our SAE and get only the activated features
    # We get a list of tuples with (index, magnitude)
    activations = self.sae.encode(x_t_squeezed)
    cleaned_activations = get_labeled_activations(activations)

    # Reset the mask so we don't choose unactivated features this pass
    self.mask = torch.zeros(self.actor.num_features)

    for activation in cleaned_activations :
      self.mask[activation[0]] = 1

    # Set to a large negative number so it comes out as 0 when passed through softmax
    masked_logits = actor_logits.clone()
    masked_logits[:, self.mask == 0] = -1e9

    # Get our log probabilities
    log_probs = F.log_softmax(masked_logits, dim=-1)

    # If we are training, we sample from the logits, else argmax
    if self.training_active : # set always true for now to test sample selection policy
      dist = torch.distributions.Categorical(logits=log_probs)
      chosen_feature = dist.sample()
    else :
      chosen_feature = torch.argmax(log_probs, dim=-1)

    self.last_feature_selection = chosen_feature.item()

    chosen_feature_log_prob = log_probs[0, chosen_feature.item()].item()

    # Get the decoder weights of the selected feature
    W_dec_feature = self.sae.W_dec[chosen_feature]

    # Apply the perturbation that will be used for the residual stream
    perturbation = self.c * W_dec_feature

    # Save a copy of the state later for training
    # The .requires_grad_(True) will be applied when loading from memory_buffer
    state = x_t_squeezed.clone()

    # Store episode information in a dictonary
    # Reward set as -1 as flag for now, will be job of training loop to update based on performance
    if self.training_active :
      self.memory_buffer.append(
          {
              "state" : state,
              "chosen_feature" : chosen_feature.item(), # <--- Use .item() to store Python scalar
              "chosen_feature_log_prob" : chosen_feature_log_prob,
              "critic_estimation" : v_s,
              "reward" : -1,
              "advantage" : 0,
              "mask" : self.mask.clone()
          }
      )

    # Unsqueeze back to original sequence length dimension before returning
    return x_t + perturbation.unsqueeze(0).unsqueeze(1)




7. Hook Wrapper Agent

While we could directly plug in our CRLController as a parameter, I chose to implement the intervention segement as a black box function so it may support different types on intervention and you just have to change the attached function, under the assumption said function takes in x_t and returens x_tilde_t.

In [ ]:
class HookWrapper :
  def __init__ (self) :
    self.modification_callback = None

  def set_mod_func (self, new_func) :
    self.modification_callback = new_func


  def hook_residual_stream(self, module, input, outputs):

    # Get residual stream from forward pass
    if isinstance(outputs, tuple):
      model_residual_stream = outputs[0]
    else:
      model_residual_stream = outputs

    residual_stream = model_residual_stream.clone()

    # Isolate the residual stream for the last token
    x_t = residual_stream[:, -1:, :]

    # Let our modifcation function make the changes to our residual stream
    if self.modification_callback is not None :
      x_tilde_t = self.modification_callback(x_t)
      residual_stream[:, -1:, :] = x_tilde_t

    if isinstance(outputs, tuple) :
      return (residual_stream,) + outputs[1:]
    return residual_stream

Below we define our actual objects in memory to use during our training loop. We also attach the hook controllers function to the models corresponding layer, which is what causes the intervention during the forward pass.

In [ ]:
actor = Actor(d_res=model_dimensions, d_sae=total_features).to(device)
critic = Critic(d_res=model_dimensions).to(device)

In [ ]:
agent_controller = CRLController(sae=sae, actor=actor, critic=critic)
interceptor_hook = HookWrapper()
interceptor_hook.set_mod_func(agent_controller.steer)

In [ ]:
# Register hook to layer 20 of the HF architecture
hook_handle = model.model.layers[20].register_forward_hook(interceptor_hook.hook_residual_stream)

8. Experiment Setup

The most valuable part of control reinforcement learning is the insights it provides to us about possible reasoning behaviour. In this block, we define directories, objects, and functions that will help us store and collect information about our agent during the training process. This helps us validate that the agent is learning meaningfull features to select from and not just randomlly choosing amplifications. This can also helps us try and trouble shoot if the agents performance is not as expected.

The below function assumes it takes in a list of tuples, where the first two elements are model_answer and target_answer. It then creates a confusion matrix for TP, TN, FP, FN. I've also included a helper function to just quickly print the returned matrix.

In [ ]:
def create_conf_matrix (scores) :
  TP = 0
  FP = 0
  TN = 0
  FN = 0

  # strip scores just to be sure

  for score in scores :
    model_ans = score[0].strip()
    target_ans = score[1].strip()

    if model_ans == "Yes" and target_ans == "Yes" :
      TP += 1
    elif model_ans == "Yes" and target_ans == "No" :
      FP += 1
    elif model_ans == "No" and target_ans == "No" :
      TN += 1
    elif model_ans == "No" and target_ans == "Yes" :
      FN += 1

  total_predictions = TP + FP + TN + FN


  return (TP, FP, TN, FN)



In [ ]:
def print_conf_matrix(matrix) :

  total_predictions = matrix[0] + matrix[1] + matrix[2] + matrix[3]

  output_string = "*----------------*\n"
  output_string += f"| TP        | {matrix[0]:>8} |\n"
  output_string += f"| FP        | {matrix[1]:>8} |\n"
  output_string += f"| FN        | {matrix[2]:>8} |\n"
  output_string += f"| TN        | {matrix[3]:>8} |\n"
  if total_predictions != 0 :
    output_string += f"| Accuracy  | {round((matrix[0] + matrix[3]) / total_predictions, 2):>8} |\n"
  else :
    output_string += f"| Accuracy  | 0 |\n"
  if matrix[0] + matrix[1] != 0 :
    output_string += f"| Precision | {round(matrix[0] / (matrix[0] + matrix[1]), 2):>8} |\n"
  else :
    output_string += f"| Precision | 0 |\n"
  if matrix[0] + matrix[2] != 0 :
    output_string += f"| Recall    | {round(matrix[0] / (matrix[0] + matrix[2]), 2):>8} |\n"
  else :
    output_string += f"| Recall    | 0 |\n"
  output_string += "*----------------*"

  print(output_string)
  return output_string

Next, I make a function to analyze the feature diversity. We want to know how many features our model is choosing from during evaluation, which features, and what semantic meanings they corrospond to. This function returns nothing and instead prints a log given a selection history.

In [ ]:
def analyze_feature_selections (selection_history, k) :
  output_string = "----------------------------------------------\n"
  output_string += f"Total Feature Selection Diversirty : {len(selection_history)}\n"
  output_string += f"Printing top {k} features ...\n"
  output_string += "----------------------------------------------\n"
  top_k = Counter(selection_history).most_common(k)

  for item in top_k :
    output_string += f"| Feature number : {item[0]} | Times selected : {item[1]}\n"

  print(output_string)

  return output_string



Now we need a evaluation function for some rough updates on the training progress.

In [ ]:
def training_eval(samples, model, CRLAgent, seed) :
  # set seeds
  random.seed(seed)


  correct_answers = 0
  accuracy = 0
  # Dict that holds {feature_id : num_times_selected} to analyze policy selection behaviour
  feature_selection = {}
  # List of tuples in form (model_answer, target_answer, subtask, sample_idx, chosen_feature, critic_estimation)
  question_results = []


  # Set agent flags
  was_training = CRLAgent.training_active
  was_steering = CRLAgent.steering_active

  CRLAgent.is_training(False)
  CRLAgent.is_steering(True)

  for sample in tqdm(range(samples), desc="Evaluating", leave=False) :
    # Pick a random question
    random_task, random_idx = random.choice(global_indices[val_end:])

    test_sample = datasets[random_task][random_idx]

    text = test_sample['text']
    target = test_sample['answer']

    prompt = f"{prompts[random_task]}\n Post : {text} \n Label : "

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=1, temperature=0.0, logits_processor=processors)

    predicted_token_id = output[0, -1]
    predicted_text_token = tokenizer.decode(predicted_token_id)

    target.lower()
    predicted_text_token.lower()

    if target in predicted_text_token :
      correct_answers += 1



    question_results.append((predicted_text_token, target, random_task, random_idx, CRLAgent.last_feature_selection, CRLAgent.last_critic_estimation))
    feature_selection[CRLAgent.last_feature_selection] = feature_selection.get(CRLAgent.last_feature_selection, 0) + 1

  # Reset to old agent flags
  CRLAgent.is_training(was_training)
  CRLAgent.is_steering(was_steering)


  accuracy = round((correct_answers / samples) * 100 , 2)
  return (correct_answers, accuracy, feature_selection, question_results)

Next, we need a benchmark evaluator. This will be used to test the whole benchmark to compare to baseline scores in categories. ( Honestly thinking, not sure if we really need this. Maybe better to do comprehensive eval with typical train_eval and just a bigger chunk of it than normal. If we eval comprehensivley on each tasks, we risk evaluating on seen training examples which may give CRL agent unfair advantage )

In [ ]:
def analyze_test_results (correct_answers, accuracy, feature_selection, question_results, k) :
  results_string = "------------------------------\n"
  results_string += f"| Overall Model Score : {accuracy}\n"
  results_string += f"| Total Questions Correct : {correct_answers}\n"
  results_string += f"| Total Questions : {len(question_results)}\n"
  results_string += "------------------------------\n"

  # Make our conf matrix
  conf_matrix = create_conf_matrix(question_results)

  # Save our strings within our print functions for logging
  conf_string = print_conf_matrix(conf_matrix)
  feature_string = analyze_feature_selections(feature_selection, k)

  results_string += conf_string
  results_string += "\n------------------------------\n"
  results_string += feature_string
  results_string += "\n------------------------------\n"

  critic_mean = np.mean([sample[5] for sample in question_results])
  critic_std = np.std([sample[5] for sample in question_results])
  results_string += f"| Critic Mean : {critic_mean}\n"
  results_string += f"| Critic Std : {critic_std}\n"

  # Now we might want to save more detailed records of questions results in a df for further analysis
  df = pd.DataFrame(columns=['Model Answer', 'Correct Answer', 'Task', 'Question', 'Selected Feature', 'Critic Value'])

  for sample in question_results :
    df.loc[len(df)] = [sample[0], sample[1], sample[2], sample[3], sample[4], sample[5]]

  return results_string, df




9. Training the agent

Checkpoint Function

In [ ]:
def checkpoint_model (crl_learner, filename) :
  # Get our file path
  filepath = os.path.join(CHECKPOINT_DIR, filename)

  # Create the overall state dict
  checkpoint = {
      'actor_network' : crl_learner.actor.state_dict(),
      'critic_network' : crl_learner.critic.state_dict(),
      'actor_optim' : crl_learner.actor_optimizer.state_dict(),
      'critic_optim' : crl_learner.critic_optimizer.state_dict(),
      'c' : crl_learner.c,
      'mask' : crl_learner.mask
  }

  torch.save(checkpoint, filepath)


def load_checkpoint (crl_learner, filepath, device) : # Changed 'filename' to 'filepath'
  if os.path.isfile(filepath):
    checkpoint = torch.load(filepath, map_location=device)

    # Load our current CRL learner with our state
    crl_learner.actor.load_state_dict(checkpoint['actor_network'])
    crl_learner.critic.load_state_dict(checkpoint['critic_network'])
    crl_learner.actor_optimizer.load_state_dict(checkpoint['actor_optim'])
    crl_learner.critic_optimizer.load_state_dict(checkpoint['critic_optim'])
    crl_learner.c = checkpoint['c']
    crl_learner.mask = checkpoint['mask']

    return crl_learner
  else :
    print(f"Warning: No checkpoint found at '{filepath}'. Starting from scratch.")

In [ ]:
PROMPT_DIR = '/content/drive/MyDrive/CRL_Learner/question_prompts/'

# Download prompts in dict
prompts = {}

# Get the prompt files ( all files under PROMPT_DIR )
for filename in ISSUE_SPOTTING_TASK :
  filepath = os.path.join(PROMPT_DIR, filename + ".txt")
  with open(filepath, 'r') as file :
    prompts[filename] = file.read()


Hyperparameters

In [ ]:
training_episodes = 4001 # How many episodes the agent trains for
train_batch_size = 64 # How often and how large the network updates occur
train_epochs = 1 # How many times we pass over the memory buffer when we train
eval_sample_size = 100 # How many tasks we evaluate the model on during evaluation
eval_period = 250 # Step interval the model is evaluated at
checkpoint_period = 250 # Period we save model parameters at
test_sample_size = 300 # Test interval for baselines and benchmarks. Should only be used after training


The last thing we do before training is preparing a file to append our results to. Storing performance information as our agent trains can help us troubleshoot and make sure our agent is learning meaningful features and converging to a optimal policy.

In [ ]:
# Create the file
results_filepath = os.path.join(CHECKPOINT_DIR, "results.txt")
with open(results_filepath, 'w') as file :
  file.write("CRL Issue Spotting Results\n")

Training Loop

In [ ]:
agent_controller.steering_active = True

for episode in tqdm(range(training_episodes) , desc="Training Process", leave=False) :
    #---PREPARE TRAIN SAMPLE---
    # Randomly select a sample from the train set of global indices
    random_task, random_idx = random.choice(global_indices[:train_end])

    # Access the sample for our training point
    train_sample = datasets[random_task][random_idx]

    text = train_sample['text']
    target = train_sample['answer']

    prompt = f"{prompts[random_task]}\n Post : {text} \n Label : "

    inputs = tokenizer(prompt, return_tensors="pt")

    # Correctly move all tensors within the inputs dictionary to the CUDA device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    #---RUN FORWARD PASS---
    # Run forward pass to trigger the hook and cache the residual stream
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=1, temperature=0.0, logits_processor=processors)

    predicted_token_id = output[0, -1]


    predicted_text_token = tokenizer.decode(predicted_token_id)

    #---PROCESS ANSWER AND UPDATES---
    target.lower()
    predicted_text_token.lower()


    if target in predicted_text_token :
      agent_controller.memory_buffer[-1]["reward"] = 1.0
    else :
      agent_controller.memory_buffer[-1]["reward"] = 0.0

    # calculate advantage
    agent_controller.memory_buffer[-1]["advantage"] = agent_controller.memory_buffer[-1]["reward"] - agent_controller.memory_buffer[-1]["critic_estimation"]

    # if eval step
    if episode % eval_period == 0  :
      print(f"Evaluating at step {episode}")
      eval_results = training_eval(eval_sample_size, model, agent_controller)

      # Extract tuple
      correct_answers = eval_results[0]
      accuracy = eval_results[1]
      feature_selection = eval_results[2]
      question_results = eval_results[3]



      results_string, df = analyze_test_results(correct_answers, accuracy, feature_selection, question_results, 15)

      # Save our results string to a bigger overall results file
      with open(results_filepath, 'a') as file :
          file.write(f"Episode {episode}\n" + results_string + '\n')


      # Save the df as a csv
      df_filename = f"question_results_{episode}.csv"
      df_filepath = os.path.join(CHECKPOINT_DIR, df_filename)
      df.to_csv(df_filepath, index=False)

    # if policy update step
    if episode % train_batch_size == 0 and episode:
      # first normalize the advantages
      adv_values = [sample["advantage"] for sample in agent_controller.memory_buffer]
      adv_mean = np.mean(adv_values)
      adv_std = np.std(adv_values)

      for sample in agent_controller.memory_buffer :
        # Add a small epsilon to adv_std to prevent division by zero
        sample["advantage"] = (sample["advantage"] - adv_mean) / (adv_std + 1e-8)

      for _ in range(train_epochs) :
        agent_controller.train()

      agent_controller.memory_buffer = []

    # if checkpoint step
    if episode % checkpoint_period == 0 and episode != 0 :
      filename = f"issue_spotting_{episode}"
      checkpoint_model(agent_controller, filename)


# Do one last save just in case at end of training
filename = f"issue_spotting_{training_episodes}_done"
checkpoint_model(agent_controller, filename)

# Save a image of our loss for our critic and our actor
png_data = agent_controller.generate_graphs()
image_filepath = os.path.join(CHECKPOINT_DIR, "run_loss_history.png")
with open(image_filepath, "wb") as f:
    f.write(png_data)

Training Process:   0%|          | 0/4001 [00:00<?, ?it/s]

Evaluating at step 0


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

*----------------*
| TP        |       50 |
| FP        |       45 |
| FN        |        5 |
| TN        |        0 |
| Accuracy  |      0.5 |
| Precision |     0.53 |
| Recall    |     0.91 |
*----------------*
----------------------------------------------
Total Feature Selection Diversirty : 5
Printing top 15 features ...
----------------------------------------------
| Feature number : 5495 | Times selected : 34
| Feature number : 1941 | Times selected : 29
| Feature number : 15729 | Times selected : 24
| Feature number : 8366 | Times selected : 9
| Feature number : 6627 | Times selected : 4



/tmp/ipykernel_6954/2046340489.py:101: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  masks = torch.stack([torch.tensor(sample["mask"], dtype=torch.bool, device=device) for sample in self.memory_buffer])


...

*----------------*
| TP        |       41 |
| FP        |       55 |
| FN        |        3 |
| TN        |        1 |
| Accuracy  |     0.42 |
| Precision |     0.43 |
| Recall    |     0.93 |
*----------------*
----------------------------------------------
Total Feature Selection Diversirty : 5
Printing top 15 features ...
----------------------------------------------
| Feature number : 5495 | Times selected : 67
| Feature number : 6627 | Times selected : 14
| Feature number : 1941 | Times selected : 8
| Feature number : 15729 | Times selected : 7
| Feature number : 8366 | Times selected : 4



/tmp/ipykernel_6954/2046340489.py:101: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  masks = torch.stack([torch.tensor(sample["mask"], dtype=torch.bool, device=device) for sample in self.memory_buffer])


Evaluating at step 4000


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

*----------------*
| TP        |       49 |
| FP        |       44 |
| FN        |        4 |
| TN        |        3 |
| Accuracy  |     0.52 |
| Precision |     0.53 |
| Recall    |     0.92 |
*----------------*
----------------------------------------------
Total Feature Selection Diversirty : 6
Printing top 15 features ...
----------------------------------------------
| Feature number : 5495 | Times selected : 61
| Feature number : 6627 | Times selected : 13
| Feature number : 1941 | Times selected : 11
| Feature number : 15729 | Times selected : 8
| Feature number : 8366 | Times selected : 6
| Feature number : 7288 | Times selected : 1



Below is a optional block to run that loads the saved parameter of a model, and then tests it based on a larger test set that is used during training evaluation. The idea is to verify over a number of iterations that the model score is due to the learned policy, and not the distribution of questions it got during the evaluation step.

The output is a table comparing all tested model scores and result and question logs from each model scored for each test being saved to a directory.

In [ ]:

test_seeds = [2004,2010,2012]
test_actor =Actor(d_res=model_dimensions, d_sae=total_features).to(device)
test_critic = Critic(d_res=model_dimensions).to(device)
test_controller = CRLController(sae=sae, actor=test_actor, critic=test_critic)
test_sample_size = 300
interceptor_hook = HookWrapper()
interceptor_hook.set_mod_func(test_controller.steer)
hook_handle = model.model.layers[20].register_forward_hook(interceptor_hook.hook_residual_stream)

base_path = '/content/drive/MyDrive/CRL_Learner/'
test_models =['issue_spotting_prompt_experiment_newtrain/issue_spotting_4000_done',
              'issue_spotting_prompt_experiment_newtrain_10reward/issue_spotting_4000_done',
              'issue_spotting_prompt_experiment_newtrain_10reward_lr5e-6/issue_spotting_4001_done',
              'issue_spotting_prompt_experiment_newtrain_lr5e-6/issue_spotting_4001_done',
              'issue_spotting_prompt_experiment_newtrain_lr5e-6_entropyloss/issue_spotting_4001_done',

              'issue_spotting_prompt_experiment_newtrain/issue_spotting_2250',
              'issue_spotting_prompt_experiment_newtrain_10reward/issue_spotting_3500',
              'issue_spotting_prompt_experiment_newtrain_10reward_lr5e-6/issue_spotting_2500',
              'issue_spotting_prompt_experiment_newtrain_lr5e-6/issue_spotting_3000',
              'issue_spotting_prompt_experiment_newtrain_lr5e-6_entropyloss/issue_spotting_3000']


model_performance = {}

TEST_DIR = '/content/drive/MyDrive/CRL_Learner/large_test_results/'
if not os.path.exists(TEST_DIR):
    os.makedirs(TEST_DIR)

for test_model in test_models :
  # this will hold the model scores on each test run
  model_performance[test_model] = []

# for each test case
for seed in test_seeds :
  # test each model
  for test_model in test_models :
    # load the model into our test controller
    checkpoint_full_path = os.path.join(base_path, test_model) # Corrected path construction
    test_controller = load_checkpoint(test_controller, checkpoint_full_path, device) # Pass full path

    # give the model to the evaluation function
    correct_answers, accuracy, feature_selection, question_results = training_eval(test_sample_size, model, test_controller, seed)

    results_str, df = analyze_test_results(correct_answers, accuracy, feature_selection, question_results, 10)

    # store the results of the evaluation function
    model_performance[test_model].append(accuracy)

    # test name
    test_name = f"{test_model.replace('/', '-')}_{seed}"
    filepath = os.path.join(TEST_DIR, test_name + ".txt")

    # Ensure the directory for the file exists
    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    with open(filepath, 'w') as file :
      file.write(f"Test results for model {test_model} : seed {seed}" + '\n')
      file.write(results_str + '\n')

    df_filename = f"question_results_{test_name}.csv"
    df_filepath = os.path.join(TEST_DIR, df_filename)
    df.to_csv(df_filepath, index=False)

for test_model in model_performance :
  print(f"Model {test_model} : {model_performance[test_model]}\n")
